In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

base_path = "/kaggle/input/datasets/xiaose/cityscapes/Cityspaces/images/val"

cities = os.listdir(base_path)

X = []
y = []

for label, city in enumerate(cities):
    folder = os.path.join(base_path, city)

    for file in os.listdir(folder):
        if file.endswith(".png"):
            img = cv2.imread(os.path.join(folder, file))
            img = cv2.resize(img, (64,64))

            X.append(img)
            y.append(label)

X = np.array(X) / 255.0
y = np.array(y)

# reduce size for speed
X = X[:400]
y = y[:400]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

LeNet

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D, Flatten, Dense, Input

model = Sequential([
    Input(shape=(64,64,3)),   #  fix warning also

    Conv2D(6, (5,5), activation='relu'),
    AveragePooling2D(pool_size=(2,2)),

    Conv2D(16, (5,5), activation='relu'),
    AveragePooling2D(pool_size=(2,2)),

    Flatten(),
    Dense(120, activation='relu'),
    Dense(84, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=15, verbose=1)

print("LeNet Accuracy:", model.evaluate(X_test, y_test)[1])

**Alexnet**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

# 🔹 One-hot encoding (same as your MNIST code)
y_train_cat = to_categorical(y_train, 3)
y_test_cat = to_categorical(y_test, 3)

# 🔹 Build AlexNet
model = Sequential()

model.add(Input(shape=(64,64,3)))

# Conv Layer 1
model.add(Conv2D(96, (5,5), strides=1, activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Conv Layer 2
model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Conv Layers 3,4,5
model.add(Conv2D(384, (3,3), activation='relu', padding='same'))
model.add(Conv2D(384, (3,3), activation='relu', padding='same'))
model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Fully connected
model.add(Flatten())

model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(3, activation='softmax'))

# 🔹 Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 🔹 Train
model.fit(
    X_train, y_train_cat,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

# 🔹 Evaluate
loss, acc = model.evaluate(X_test, y_test_cat)

print("AlexNet Accuracy:", acc)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

# one-hot
y_train_cat = to_categorical(y_train, 3)
y_test_cat = to_categorical(y_test, 3)

model = Sequential([
    Input(shape=(64,64,3)),

    Conv2D(32, (3,3), activation='relu', padding='same'),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),

    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train_cat, epochs=15, batch_size=32, validation_data=(X_test, y_test_cat))

print("VGG Accuracy:", model.evaluate(X_test, y_test_cat)[1])

ZFNet

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical

# one-hot
y_train_cat = to_categorical(y_train, 3)
y_test_cat = to_categorical(y_test, 3)

model = Sequential([
    Input(shape=(64,64,3)),

    Conv2D(96, (7,7), strides=2, activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(256, (5,5), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Conv2D(384, (3,3), activation='relu', padding='same'),
    Conv2D(384, (3,3), activation='relu', padding='same'),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),

    Flatten(),

    Dense(512, activation='relu'),
    Dropout(0.5),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train, y_train_cat,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test_cat)
)

print("ZF-Net Accuracy:", model.evaluate(X_test, y_test_cat)[1])

FINAL OBSERVATIONS

* LeNet achieved the highest accuracy (~83.7%) due to its simple architecture, which is well-suited for small datasets.

* AlexNet showed slightly lower accuracy (~80%) but performed well due to deeper layers and better feature extraction.

* VGG model achieved moderate accuracy (~72.5%) as deeper architectures require more data and are harder to train from scratch.

* ZF-Net performed poorly (~36%) due to excessive complexity and reduction in feature map size, leading to ineffective learning on a small dataset.

Week8

GoogleNet

In [31]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, Input, Concatenate, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# one-hot
y_train_cat = to_categorical(y_train, 3)
y_test_cat = to_categorical(y_test, 3)

# 🔹 Inception block
def inception_block(x):
    path1 = Conv2D(32, (1,1), activation='relu', padding='same')(x)

    path2 = Conv2D(32, (1,1), activation='relu', padding='same')(x)
    path2 = Conv2D(64, (3,3), activation='relu', padding='same')(path2)

    path3 = Conv2D(32, (1,1), activation='relu', padding='same')(x)
    path3 = Conv2D(64, (5,5), activation='relu', padding='same')(path3)

    path4 = MaxPooling2D((3,3), strides=1, padding='same')(x)
    path4 = Conv2D(32, (1,1), activation='relu', padding='same')(path4)

    return Concatenate()([path1, path2, path3, path4])

# 🔹 Model
input_layer = Input(shape=(64,64,3))

x = Conv2D(32, (3,3), activation='relu', padding='same')(input_layer)
x = MaxPooling2D((2,2))(x)

x = inception_block(x)
x = MaxPooling2D((2,2))(x)

x = inception_block(x)

x = AveragePooling2D((2,2))(x)
x = Flatten()(x)

x = Dense(128, activation='relu')(x)
output = Dense(3, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train_cat, epochs=15, batch_size=32, validation_data=(X_test, y_test_cat))

print("GoogLeNet Accuracy:", model.evaluate(X_test, y_test_cat)[1])

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 457ms/step - accuracy: 0.3590 - loss: 1.0414 - val_accuracy: 0.3875 - val_loss: 0.9853
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 407ms/step - accuracy: 0.5983 - loss: 0.9127 - val_accuracy: 0.6500 - val_loss: 0.8272
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 423ms/step - accuracy: 0.7178 - loss: 0.7020 - val_accuracy: 0.6500 - val_loss: 0.7898
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 406ms/step - accuracy: 0.7329 - loss: 0.6548 - val_accuracy: 0.5875 - val_loss: 0.9762
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 406ms/step - accuracy: 0.7686 - loss: 0.6090 - val_accuracy: 0.7125 - val_loss: 0.8516
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 400ms/step - accuracy: 0.7728 - loss: 0.5144 - val_accuracy: 0.7125 - val_loss: 0.6967
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 415ms/step - accuracy: 0.8167 - loss: 0.4419 - val_accuracy: 0.6500 - val_loss: 0.7748
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 400ms/step - accuracy: 0.8274 - loss: 0.3695 - val_accuracy: 0.

ResNet

In [32]:
from tensorflow.keras.layers import Add

# 🔹 Residual block
def res_block(x, filters):
    shortcut = x

    x = Conv2D(filters, (3,3), padding='same', activation='relu')(x)
    x = Conv2D(filters, (3,3), padding='same')(x)

    x = Add()([x, shortcut])   # 🔥 skip connection
    x = tf.keras.layers.Activation('relu')(x)

    return x

# 🔹 Model
input_layer = Input(shape=(64,64,3))

x = Conv2D(32, (3,3), activation='relu', padding='same')(input_layer)

x = res_block(x, 32)
x = res_block(x, 32)

x = MaxPooling2D((2,2))(x)

x = res_block(x, 32)

x = AveragePooling2D((2,2))(x)
x = Flatten()(x)

x = Dense(128, activation='relu')(x)
output = Dense(3, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train_cat, epochs=15, batch_size=32, validation_data=(X_test, y_test_cat))

print("ResNet Accuracy:", model.evaluate(X_test, y_test_cat)[1])

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 660ms/step - accuracy: 0.4694 - loss: 1.3024 - val_accuracy: 0.6000 - val_loss: 0.9329
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 621ms/step - accuracy: 0.6033 - loss: 0.9341 - val_accuracy: 0.4375 - val_loss: 0.9777
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 648ms/step - accuracy: 0.6007 - loss: 0.8413 - val_accuracy: 0.7125 - val_loss: 0.7635
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 625ms/step - accuracy: 0.7866 - loss: 0.6144 - val_accuracy: 0.7000 - val_loss: 0.7147
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 655ms/step - accuracy: 0.7553 - loss: 0.5816 - val_accuracy: 0.7000 - val_loss: 0.7821
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 653ms/step - accuracy: 0.7756 - loss: 0.5220 - val_accuracy: 0.7375 - val_loss: 0.6489
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 627ms/step - accuracy: 0.8831 - loss: 0.3732 - val_accuracy: 0.7000 - val_loss: 0.7650
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 615ms/step - accuracy: 0.8616 - loss: 0.2917 - val_accuracy: 

FINAL OBSERVATIONS

* GoogLeNet achieved moderate accuracy (~76.2%) by using Inception blocks, which capture features at multiple scales.

* ResNet achieved higher accuracy (~78.7%) due to the use of skip connections, which improve gradient flow and enable better learning.

* ResNet outperformed GoogLeNet as residual connections help in training deeper networks more effectively.